# Blood Transfusion Service Classification

748 blood donors, each described by four numbers: months since the last donation (Recency), total number of donations (Frequency), total blood donated in c.c. (Monetary), and months since the first donation (Time). Predict whether the donor gave blood again in March 2007.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Blood Transfusion Service Center (id 176)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/176/data.csv')))
print(df.shape)
print('Donors per outcome (1 = donated again, 0 = did not):')
print(df['Donated_Blood'].value_counts())
df.head(3)


(748, 5)
Donors per outcome (1 = donated again, 0 = did not):
Donated_Blood
0    570
1    178
Name: count, dtype: int64


,Recency,Frequency,Monetary,Time,Donated_Blood
0,2,50,12500,98,1
1,0,13,3250,28,1
2,1,16,4000,35,1


In [2]:

feats = ['Recency', 'Frequency', 'Monetary', 'Time']
X = df[feats]; y = df['Donated_Blood']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.7059
Logistic Regression accuracy: 0.7701
              precision    recall  f1-score   support

           0       0.80      0.83      0.81       143
           1       0.36      0.32      0.34        44

    accuracy                           0.71       187
   macro avg       0.58      0.57      0.57       187
weighted avg       0.69      0.71      0.70       187



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp.head(20))
imp.head(20).plot(kind='barh', title='Which donation-history numbers matter most?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


Time         0.434633
Recency      0.260601
Frequency    0.155059
Monetary     0.149706
dtype: float64


Winner: Logistic Regression
